## Setup
Install dependencies and download competition data. On Kaggle with internet disabled, skip the install cells and use preinstalled packages or attach wheel files as a Dataset.

In [ ]:
!kaggle competitions download -c csiro-biomass
!unzip csiro-biomass.zip
!rm csiro-biomass.zip

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
# transformers installed from source because DINOv3 support is not in the stable release yet
!pip install -q git+https://github.com/huggingface/transformers.git accelerate timm
!pip install -Uq huggingface_hub
!pip install -q lightgbm catboost albumentations opencv-python-headless scikit-learn tqdm

# after the first run of this cell, restart the runtime then rerun from here

## Load data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.config import cfg, DEVICE, ALL_TARGETS, R2_WEIGHTS
from src.data import load_dataframes, attach_absolute_paths
from src.metrics import weighted_r2_global, enforce_mass_balance
from src.features import (
    compute_siglip_embeddings, compute_semantic_features,
    EmbeddingFeaturizer, siglip_oof_predict,
)
from src.models import train_dino_oof, train_dino_full
from src.gate import train_gate

import numpy as np
import pandas as pd

print('DEVICE:', DEVICE)

In [ ]:
train_wide, test_long = load_dataframes()
train_wide = attach_absolute_paths(train_wide)

if cfg.FAST_DEBUG:
    train_wide = train_wide.sample(
        min(cfg.DEBUG_SAMPLES, len(train_wide)), random_state=cfg.SEED
    ).reset_index(drop=True)

print('train_wide:', train_wide.shape)
print('test_long :', test_long.shape)

## SigLIP branch
Compute patch-averaged image embeddings, zero-shot semantic features, then train boosting regressors out-of-fold.

In [ ]:
train_embeddings = compute_siglip_embeddings(
    train_wide['abs_path'].tolist(),
    cfg.CACHE_DIR / 'siglip_train_emb.npy',
)
print(f'train_embeddings: {train_embeddings.shape}')

In [ ]:
train_semantic = compute_semantic_features(
    train_embeddings,
    cfg.CACHE_DIR / 'siglip_train_sem.npy',
)
print(f'train_semantic: {train_semantic.shape}')

In [ ]:
ground_truth = train_wide[ALL_TARGETS].values.astype(np.float32)

oof_siglip = siglip_oof_predict(
    train_embeddings, train_semantic, ground_truth,
    n_folds=cfg.N_FOLDS,
)
print(f'oof_siglip: {oof_siglip.shape}')
print(f'SigLIP out-of-fold R2: {weighted_r2_global(ground_truth, oof_siglip):.4f}')

## DINOv3 branch
Fine-tune DINOv3 with K-fold cross-validation to produce out-of-fold predictions.

In [ ]:
oof_dino = train_dino_oof(train_wide, n_folds=cfg.N_FOLDS)
print(f'oof_dino: {oof_dino.shape}')
print(f'DINOv3 out-of-fold R2: {weighted_r2_global(ground_truth, oof_dino):.4f}')

## Stacking gate
Train per-target logistic regression classifiers that decide, sample by sample, whether to trust DINOv3 or SigLIP more.

In [ ]:
gate_models, oof_blended = train_gate(
    oof_siglip, oof_dino, ground_truth,
    train_semantic, None, train_wide,
)

## Full-data DINOv3 retrain
Retrain on all data with Automatic Mixed Precision, Exponential Moving Average, and MixUp for the final submission model.

In [ ]:
model_dino, ema_model = train_dino_full(train_wide, ground_truth)

## Save artifacts

In [ ]:
import pickle
import json as _json
import shutil
from pathlib import Path

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

import torch

SAVE_DIR = Path('/content/models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1/6 SigLIP feature engine
print('[1/6] SigLIP feature engine')
engine_full = EmbeddingFeaturizer(pca_var=0.80, n_pls=8, n_clusters=6, seed=cfg.SEED)
engine_full.fit(train_embeddings, y=ground_truth)
with open(SAVE_DIR / 'siglip_feature_engine.pkl', 'wb') as f:
    pickle.dump(engine_full, f)

# 2/6 SigLIP boosting models retrained on full data
print('[2/6] SigLIP boosting models')
boost_dir = SAVE_DIR / 'siglip_boosting'
boost_dir.mkdir(exist_ok=True)

catboost_parameters = dict(iterations=1200, learning_rate=0.05, depth=4,
                           random_state=cfg.SEED, verbose=0, allow_writing_files=False)
lightgbm_parameters = dict(n_estimators=700, learning_rate=0.025, num_leaves=48,
                           subsample=0.75, colsample_bytree=0.75, random_state=cfg.SEED, verbose=-1)
histgb_parameters = dict(max_iter=350, learning_rate=0.05, random_state=cfg.SEED)

feature_matrix = engine_full.transform(train_embeddings, train_semantic)

for target_index, target_name in enumerate(ALL_TARGETS):
    print(f'  {target_name}')
    cb = CatBoostRegressor(**catboost_parameters)
    cb.fit(feature_matrix, ground_truth[:, target_index])
    cb.save_model(str(boost_dir / f'{target_name}_catboost.cbm'))

    lgb = LGBMRegressor(**lightgbm_parameters)
    lgb.fit(feature_matrix, ground_truth[:, target_index])
    lgb.booster_.save_model(str(boost_dir / f'{target_name}_lightgbm.txt'))

    hgb = HistGradientBoostingRegressor(**histgb_parameters)
    hgb.fit(feature_matrix, ground_truth[:, target_index])
    with open(boost_dir / f'{target_name}_histgb.pkl', 'wb') as f:
        pickle.dump(hgb, f)

# 3/6 DINOv3 checkpoint (prefer Exponential Moving Average weights)
print('[3/6] DINOv3 checkpoint')
source_model = ema_model.module
hidden_dimension = getattr(source_model, 'hidden_dim', 768)

torch.save({
    'model_state_dict': source_model.state_dict(),
    'hidden_dim': hidden_dimension,
    'config': {'model_name': cfg.DINO_MODEL_NAME, 'img_size': cfg.IMG_SIZE},
}, SAVE_DIR / 'dinov3_regressor.pth')

torch.save({
    'film':     source_model.film.state_dict(),
    'h_green':  source_model.h_green.state_dict(),
    'h_clover': source_model.h_clover.state_dict(),
    'h_dead':   source_model.h_dead.state_dict(),
    'hidden_dim': hidden_dimension,
}, SAVE_DIR / 'dinov3_heads_only.pth')

# 4/6 gate classifiers
print('[4/6] gate classifiers')
with open(SAVE_DIR / 'gate_models.pkl', 'wb') as f:
    pickle.dump(gate_models, f)

# 5/6 configuration metadata
print('[5/6] configuration')
with open(SAVE_DIR / 'config.json', 'w') as f:
    _json.dump({
        'ALL_TARGETS': ALL_TARGETS,
        'R2_WEIGHTS': R2_WEIGHTS.tolist(),
        'SIGLIP_PATH': cfg.SIGLIP_PATH,
        'DINO_MODEL_NAME': cfg.DINO_MODEL_NAME,
        'IMG_SIZE': cfg.IMG_SIZE,
        'PATCH_SIZE': cfg.PATCH_SIZE,
        'OVERLAP': cfg.OVERLAP,
        'SIGLIP_EMB_DIM': int(train_embeddings.shape[1]),
        'SEED': cfg.SEED,
    }, f, indent=2)

# 6/6 cached embedding files
print('[6/6] cached embeddings')
for filename in ['siglip_train_emb.npy', 'siglip_train_sem.npy']:
    source_path = cfg.CACHE_DIR / filename
    if source_path.exists():
        shutil.copy(source_path, SAVE_DIR / filename)
        print(f'  copied {filename}')

# summary
print()
total_megabytes = 0.0
for entry in sorted(SAVE_DIR.iterdir()):
    if entry.is_file():
        size_megabytes = entry.stat().st_size / (1024 * 1024)
        total_megabytes += size_megabytes
        print(f'  {entry.name}: {size_megabytes:.2f} MB')
    else:
        directory_size = sum(x.stat().st_size for x in entry.rglob('*') if x.is_file()) / (1024 * 1024)
        total_megabytes += directory_size
        print(f'  {entry.name}/: {directory_size:.2f} MB')
print(f'  TOTAL: {total_megabytes:.2f} MB')

## Download pretrained weights
Run these cells in Colab to download and zip the Hugging Face model weights for offline use on Kaggle.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='google/siglip-base-patch16-224',
    local_dir='/content/siglip-base-patch16-224',
    local_dir_use_symlinks=False,
)
!zip -r /content/siglip-base-patch16-224.zip /content/siglip-base-patch16-224

from google.colab import files
files.download('/content/siglip-base-patch16-224.zip')

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='facebook/dinov3-vitb16-pretrain-lvd1689m',
    local_dir='/content/dinov3-vitb16-pretrain-lvd1689m',
    local_dir_use_symlinks=False,
)
!zip -r /content/dinov3-vitb16-pretrain-lvd1689m.zip /content/dinov3-vitb16-pretrain-lvd1689m

from google.colab import files
files.download('/content/dinov3-vitb16-pretrain-lvd1689m.zip')